# Orchestrator-Workers: el manager decide sobre la marcha

Clasificación: **Proceso jerárquico.** Un manager LLM lee la petición del cliente y decide qué agentes necesita, en qué orden y cuánto trabajo darle a cada uno.

A diferencia de parallelization (donde las 4 tasks están fijadas antes de arrancar), aquí el manager adapta la ejecución al caso concreto. Un cliente que solo quiere relajarse necesita más trabajo de actividades; uno con itinerario ajustado, más de vuelos.

## Cómo funciona en CrewAI

`Process.hierarchical` activa un manager automático que orquesta a los agentes. El manager recibe la task principal, decide a quién delegar, y puede volver a consultar al mismo agente si necesita más detalle.

```python
crew = Crew(
    agents=[...],
    tasks=[main_task],
    process=Process.hierarchical,
    manager_llm="gpt-4o-mini",
)
```

In [4]:
!uv pip install -r requirements.txt --quiet

In [5]:
from dotenv import load_dotenv
import nest_asyncio

load_dotenv()
nest_asyncio.apply()

In [6]:
from crewai import Agent, Task, Crew, Process
from viajes_crew import ViajesCrew

base_crew = ViajesCrew()

peticion_cliente = (
    "Viaje a Islandia, 5 dias, 2 personas, 2200 EUR en total. "
    "Lo unico que de verdad importa es ver auroras boreales y banarnos en fuentes termales; "
    "el resto (vuelos, alojamiento) lo he revisado ya manualmente."
)

main_task = Task(
    description=(
        f"Peticion del cliente: {peticion_cliente}\n\n"
        "Decide que especialistas necesitas y en que orden, dado las prioridades del cliente. "
        "Entrega un itinerario completo y el presupuesto desglosado."
    ),
    expected_output="Itinerario dia a dia con vuelos, alojamiento, actividades y transporte, coste por partida y total dentro del presupuesto indicado.",
    agent=base_crew.actividades(),
)

crew = Crew(
    agents=[base_crew.vuelos(), base_crew.alojamiento(), base_crew.actividades(), base_crew.transporte()],
    tasks=[main_task],
    process=Process.hierarchical,
    manager_llm="gpt-4o-mini",
    verbose=True,
)

result = await crew.kickoff_async()
print(result.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: a5229956-8f95-40a2-8a0a-b46a1f24c6e5                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Peticion del cliente: Viaje a Islandia, 5 dias, 2 personas, 2200 EUR en total. Lo unico que de verdad    │
│  importa es ver auroras boreales y banarnos en fuentes termales; el resto (vuelos, alojamiento) lo he revisado  │
│  ya manualmente.                                                                                                │
│                                                                                                                 │
│  Decide que especialistas necesitas y en que orden, dado las prioridades del cliente. Entrega un itinerario     │
│  completo y el presupuesto desglosado.                                                                          │
│  ID: 327f2b80-887f-4e38-bb47-1731154d6567                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Task: Peticion del cliente: Viaje a Islandia, 5 dias, 2 personas, 2200 EUR en total. Lo unico que de verdad    │
│  importa es ver auroras boreales y banarnos en fuentes termales; el resto (vuelos, alojamiento) lo he revisado  │
│  ya manualmente.                                                                                                │
│                                                                                                                 │
│  Decide que especialistas necesitas y en que orden, dado las prioridades del cliente. Entrega un itinerario     │
│  completo y el presupuesto desglosado.                                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#7) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: delegate_work_to_coworker                                                                                │
│  Args: {'task': 'Identify and recommend thermal hot springs suitable for relaxation during the trip. Include    │
│  information about entry fees, amenities, and transportation options.', 'context': 'The client want...          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Event pairing mismatch. 'tool_usage_finished' closed 'agent_execution_started' (expected
'tool_usage_started')

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Itinerario para un viaje de 5 días a Islandia: 2 personas, presupuesto de 2200 EUR**                         │
│                                                                                                                 │
│  ### **Día 1: Llegada a Reikiavik**                                                                             │
│  - **Transporte**: Vuelo hacia el Aeropuerto Internacional de Keflavik (KEF). (Verificar el costo total del     │
│  vuelo en la búsqueda).                                                                                         │
│  - **Alojamiento**: Reservar en un hotel o Airbnb en Reikiavik (presupuesto ~ 100-150 EUR por noche).           │
│  - **Actividad**: Explorar el centro de Reikiavik: Hallgrímskirkja, Laugavegur (calle principal de compras) y   │
│  cena en un restaurante local (~ 50-70 EUR en total).                                                           │
│  - **Costo total del día**: (Alojamiento: 150 EUR, Comida: 70 EUR) = **220 EUR**                                │
│                                                                                                                 │
│  ### **Día 2: Blue Lagoon y Reykjanes Peninsula**                                                               │
│  - **Desayuno**: Incluido en el alojamiento.                                                                    │
│  - **Visita**: Blue Lagoon.                                                                                     │
│    - **Entrada**: Aproximadamente *6,900 ISK* (~ 45 EUR).                                                       │
│    - **Transporte**: Alquiler de coche o traslado desde Reikiavik es recomendable (~ 40 EUR por                 │
│  gasolina/taxi).                                                                                                │
│  - **Almuerzo**: Restaurante en Blue Lagoon (~ 50 EUR).                                                         │
│  - **Tarde**: Regresar a Reikiavik y explorar el puerto y la zona de Harpa.                                     │
│  - **Cena**: Restaurante local (~ 50 EUR).                                                                      │
│  - **Costo total del día**: (Blue Lagoon: 45 EUR, Transporte: 40 EUR, Almuerzo: 50 EUR, Cena: 50 EUR) = **225   │
│  EUR**                                                                                                          │
│                                                                                                                 │
│  ### **Día 3: Golden Circle**                                                                                   │
│  - **Desayuno**: Incluido en el alojamiento.                                                                    │
│  - **Ruta**: Visitar Thingvellir, Geysir y Gulfoss.                                                             │
│  - **Almuerzo**: Picnic o cena rápida en una de las paradas (~ 30 EUR).                                         │
│  - **Actividad nocturna**: Excursión organizada para ver las Auroras Boreales.                                  │
│    - **Costo**: Alrededor de *8,000 ISK* (~ 50 EUR por persona).                                                │
│  - **Costo total del día**: (Transporte: 40 EUR, Almuerzo: 30 EUR, Cena: 50 EUR, Tour: 100 EUR) = **220 EUR**   │
│                                                        

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Peticion del cliente: Viaje a Islandia, 5 dias, 2 personas, 2200 EUR en total. Lo unico que de verdad    │
│  importa es ver auroras boreales y banarnos en fuentes termales; el resto (vuelos, alojamiento) lo he revisado  │
│  ya manualmente.                                                                                                │
│                                                                                                                 │
│  Decide que especialistas necesitas y en que orden, dado las prioridades del cliente. Entrega un itinerario     │
│  completo y el presupuesto desglosado.                                                                          │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: a5229956-8f95-40a2-8a0a-b46a1f24c6e5                                                                       │
│  Final Output: **Itinerario para un viaje de 5 días a Islandia: 2 personas, presupuesto de 2200 EUR**           │
│                                                                                                                 │
│  ### **Día 1: Llegada a Reikiavik**                                                                             │
│  - **Transporte**: Vuelo hacia el Aeropuerto Internacional de Keflavik (KEF). (Verificar el costo total del     │
│  vuelo en la búsqueda).                                                                                         │
│  - **Alojamiento**: Reservar en un hotel o Airbnb en Reikiavik (presupuesto ~ 100-150 EUR por noche).           │
│  - **Actividad**: Explorar el centro de Reikiavik: Hallgrímskirkja, Laugavegur (calle principal de compras) y   │
│  cena en un restaurante local (~ 50-70 EUR en total).                                                           │
│  - **Costo total del día**: (Alojamiento: 150 EUR, Comida: 70 EUR) = **220 EUR**                                │
│                                                                                                                 │
│  ### **Día 2: Blue Lagoon y Reykjanes Peninsula**                                                               │
│  - **Desayuno**: Incluido en el alojamiento.                                                                    │
│  - **Visita**: Blue Lagoon.                                                                                     │
│    - **Entrada**: Aproximadamente *6,900 ISK* (~ 45 EUR).                                                       │
│    - **Transporte**: Alquiler de coche o traslado desde Reikiavik es recomendable (~ 40 EUR por                 │
│  gasolina/taxi).                                                                                                │
│  - **Almuerzo**: Restaurante en Blue Lagoon (~ 50 EUR).                                                         │
│  - **Tarde**: Regresar a Reikiavik y explorar el puerto y la zona de Harpa.                                     │
│  - **Cena**: Restaurante local (~ 50 EUR).                                                                      │
│  - **Costo total del día**: (Blue Lagoon: 45 EUR, Transporte: 40 EUR, Almuerzo: 50 EUR, Cena: 50 EUR) = **225   │
│  EUR**                                                                                                          │
│                                                                                                                 │
│  ### **Día 3: Golden Circle**                                                                                   │
│  - **Desayuno**: Incluido en el alojamiento.                                                                    │
│  - **Ruta**: Visitar Thingvellir, Geysir y Gulfoss.                                                             │
│  - **Almuerzo**: Picnic o cena rápida en una de las paradas (~ 30 EUR).                                         │
│  - **Actividad nocturna**: Excursión organizada para ver las Auroras Boreales.                                  │
│    - **Costo**: Alrededor de *8,000 ISK* (~ 50 EUR por persona).                                                │
│  - **Costo total del día**: (Transporte: 40 EUR, Almuerzo: 30 EUR, Cena: 50 EUR, Tour: 100 EUR) = **220 EUR**   │
│                                                       

**Itinerario para un viaje de 5 días a Islandia: 2 personas, presupuesto de 2200 EUR**

### **Día 1: Llegada a Reikiavik**
- **Transporte**: Vuelo hacia el Aeropuerto Internacional de Keflavik (KEF). (Verificar el costo total del vuelo en la búsqueda).
- **Alojamiento**: Reservar en un hotel o Airbnb en Reikiavik (presupuesto ~ 100-150 EUR por noche).
- **Actividad**: Explorar el centro de Reikiavik: Hallgrímskirkja, Laugavegur (calle principal de compras) y cena en un restaurante local (~ 50-70 EUR en total).
- **Costo total del día**: (Alojamiento: 150 EUR, Comida: 70 EUR) = **220 EUR**

### **Día 2: Blue Lagoon y Reykjanes Peninsula**
- **Desayuno**: Incluido en el alojamiento.
- **Visita**: Blue Lagoon.
  - **Entrada**: Aproximadamente *6,900 ISK* (~ 45 EUR).
  - **Transporte**: Alquiler de coche o traslado desde Reikiavik es recomendable (~ 40 EUR por gasolina/taxi).
- **Almuerzo**: Restaurante en Blue Lagoon (~ 50 EUR).
- **Tarde**: Regresar a Reikiavik y explorar el puerto y la 

╭────────────────────────── Trace Batch Finalization ──────────────────────────╮
│ ✅ Trace batch finalized with session ID:                                    │
│ bb0427da-bee8-4806-ab10-74d4761b58f9                                         │
│                                                                              │
│ 🔗 View here:                                                                │
│ https://app.crewai.com/crewai_plus/trace_batches/bb0427da-bee8-4806-ab10-74d │
│ 4761b58f9                                                                    │
╰──────────────────────────────────────────────────────────────────────────────╯


## Qué define este patrón

El manager decide en runtime cuánto delega a cada agente. Puede consultar poco a transporte y volver dos veces a actividades si la petición lo requiere.